<a href="https://colab.research.google.com/github/Fahad-Hafeez/Cybersecurity-and-Financial-Tabular-Benchmarks-ml-classifier-comparison/blob/main/Significance_Aware_Evaluation_of_Classical_and_Transformer_Classifiers_Across_Cybersecurity_and_Financial_Tabular_Benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set up Google Drive Data Directory

Assuming the dataset files (`Training Dataset.arff`, `KDDTrain+.txt`, `KDDTest+.txt`, `spambase.data`, `creditcard.csv`) are located in a folder named `datasets` within your Google Drive's `MyDrive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/R2 Datasets'
print(f"Data directory set to: {DATA_DIR}")

In [ ]:
import os
print(f"Contents of {DATA_DIR}:")
for item in os.listdir(DATA_DIR):
    print(item)

In [ ]:
!pip install imbalanced-learn pingouin -q
!pip install statsmodels -q

import numpy as np
import pandas as pd
import warnings, json, os, time
from scipy.io import arff

from sklearn.preprocessing import (StandardScaler, OrdinalEncoder,
                                    LabelEncoder)
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                      train_test_split, cross_val_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (f1_score, accuracy_score,
                               precision_score, recall_score,
                               confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from statsmodels.stats.contingency_tables import mcnemar as mc_test
import pingouin as pg
import scipy.stats as ss
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print("All imports OK")

In [ ]:
NSL_COLS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files',
    'num_outbound_cmds','is_host_login','is_guest_login','count',
    'srv_count','serror_rate','srv_serror_rate','rerror_rate',
    'srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty_level'
]
NSL_CAT_COLS = ['protocol_type', 'service', 'flag']

In [ ]:
def load_phishing(path='PhishingData.arff'):
    raw, meta = arff.loadarff(path)
    df = pd.DataFrame(raw)
    df = df.applymap(lambda x: x.decode() if isinstance(x,bytes) else x)
    df = df.astype(float)
    X = df.iloc[:, :-1].values
    # Original labels: 1=phishing, 0=suspicious, -1=legitimate
    # Remap: phishing(1) → 1, others → 0
    y = (df.iloc[:, -1].values == 1).astype(int)
    print(f"Phishing: X={X.shape}, y={np.bincount(y)}")
    return X, y

def load_nslkdd(train_path='KDDTrain+.txt', test_path='KDDTest+.txt'):
    oe = OrdinalEncoder(handle_unknown='use_encoded_value',
                         unknown_value=-1)
    def _load(path, fit_oe=False):
        df = pd.read_csv(path, names=NSL_COLS)
        df = df.drop('difficulty_level', axis=1)
        if fit_oe:
            df[NSL_CAT_COLS] = oe.fit_transform(df[NSL_CAT_COLS])
        else:
            df[NSL_CAT_COLS] = oe.transform(df[NSL_CAT_COLS])
        X = df.drop('label', axis=1).values.astype(float)
        y = (df['label'] != 'normal').astype(int).values
        return X, y
    X_tr, y_tr = _load(train_path, fit_oe=True)
    X_te, y_te = _load(test_path,  fit_oe=False)
    print(f"NSL-KDD train: {X_tr.shape}, test: {X_te.shape}")
    print(f"  Train balance: {np.bincount(y_tr)}")
    print(f"  Test balance:  {np.bincount(y_te)}")
    return X_tr, y_tr, X_te, y_te

def load_spambase(path='spambase.data'):
    df = pd.read_csv(path, header=None)
    X = df.iloc[:, :57].values.astype(float)
    y = df.iloc[:, 57].values.astype(int)
    print(f"Spambase: X={X.shape}, y={np.bincount(y)}")
    return X, y

def load_creditcard(path='creditcard.csv'):
    df = pd.read_csv(path, low_memory=False)
    df = df.drop('Time', axis=1)
    X = df.drop('Class', axis=1).values.astype(float)
    y = df['Class'].values.astype(int)
    print(f"Credit Card: X={X.shape}, y={np.bincount(y)}")
    return X, y

In [ ]:
SMOTE_CAP = 20_000   # max rows fed into SMOTE — fast on any dataset

def make_splits(X, y, test_size=0.20, val_frac=0.20,
                apply_smote=False, apply_pca=False,
                pca_var=0.95, seed=SEED,
                X_te_fixed=None, y_te_fixed=None,
                smote_cap=SMOTE_CAP):         # ← new parameter
    if X_te_fixed is not None:
        X_te, y_te = X_te_fixed.copy(), y_te_fixed.copy()
        X_tv, y_tv = X.copy(), y.copy()
        val_actual = val_frac / (1 - test_size)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tv, y_tv, test_size=val_actual,
            stratify=y_tv, random_state=seed)
    else:
        X_tv, X_te, y_tv, y_te = train_test_split(
            X, y, test_size=test_size,
            stratify=y, random_state=seed)
        val_actual = val_frac / (1 - test_size)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tv, y_tv, test_size=val_actual,
            stratify=y_tv, random_state=seed)

    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    X_te  = scaler.transform(X_te)

    if apply_smote:
        # --- SMOTE CAP: subsample before SMOTE to control KNN cost ---
        if smote_cap and len(X_tr) > smote_cap:
            rng = np.random.RandomState(seed)
            idx = rng.choice(len(X_tr), smote_cap, replace=False)
            X_smote, y_smote = X_tr[idx], y_tr[idx]
            print(f"    SMOTE cap: {len(X_tr):,} → {smote_cap:,} rows")
        else:
            X_smote, y_smote = X_tr, y_tr
        sm = SMOTE(random_state=seed, k_neighbors=5)
        X_tr, y_tr = sm.fit_resample(X_smote, y_smote)
        print(f"    After SMOTE: {np.bincount(y_tr)}")

    pca_obj = None
    if apply_pca:
        pca_obj = PCA(n_components=pca_var, random_state=seed)
        X_tr  = pca_obj.fit_transform(X_tr)
        X_val = pca_obj.transform(X_val)
        X_te  = pca_obj.transform(X_te)
        print(f"    PCA: {X_tr.shape[1]} components")

    return X_tr, X_val, X_te, y_tr, y_val, y_te, scaler, pca_obj

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (StratifiedKFold, RandomizedSearchCV,
                                      GridSearchCV)
import time

SEED = 42

# -------------------------------------------------------
# Dataset size thresholds
# -------------------------------------------------------
LARGE_THRESHOLD = 10_000   # use LinearSVC + saga LR above this
HUGE_THRESHOLD  = 50_000   # also reduce RF n_estimators above this

def get_clf_configs(n_train):
    """
    Return classifier configs appropriate for training set size.
    Smaller grids and faster solvers for large datasets.
    """
    is_large = n_train > LARGE_THRESHOLD
    is_huge  = n_train > HUGE_THRESHOLD

    # --- Logistic Regression ---
    # saga: stochastic, much faster on large datasets
    # lbfgs: more accurate on small datasets
    lr_solver  = 'saga'  if is_large else 'lbfgs'
    lr_config  = {
        'estimator': LogisticRegression(
            solver=lr_solver, max_iter=1000,
            random_state=SEED, n_jobs=-1),
        'param_grid': {'C': [0.01, 0.1, 1, 10]},  # 4 values not 6
        'search': 'grid'
    }

    # --- Random Forest ---
    # Fewer estimators on huge datasets; randomised search
    if is_huge:
        rf_grid   = {
            'n_estimators': [200],
            'max_depth':    [10, 20],
        }
        rf_search = 'grid'   # only 4 combos, grid is fine
    else:
        rf_grid   = {
            'n_estimators': [100, 200, 300],
            'max_depth':    [None, 10, 20],
            'min_samples_leaf': [1, 2]
        }
        rf_search = 'random'  # 18 combos → sample 8
    rf_config = {
        'estimator': RandomForestClassifier(
            random_state=SEED, n_jobs=-1),
        'param_grid': rf_grid,
        'search': rf_search,
        'n_iter': 8   # used only when search='random'
    }

    # --- SVM ---
    # LinearSVC for large datasets (linear kernel, much faster)
    # Wrap in CalibratedClassifierCV so .predict_proba() works
    # SVC(rbf) only for small datasets
    if is_large:
        svm_config = {
            'estimator': CalibratedClassifierCV(
                LinearSVC(max_iter=2000, random_state=SEED), cv=3),
            'param_grid': {'estimator__C': [0.1, 1, 10]},
            'search': 'grid'
        }
    else:
        svm_config = {
            'estimator': SVC(
                kernel='rbf', gamma='scale',
                random_state=SEED, cache_size=2000),
            'param_grid': {'C': [0.1, 1, 10, 100]},
            'search': 'grid'
        }

    return {'LR': lr_config, 'RF': rf_config, 'SVM': svm_config}


def get_cv(n_train):
    """3 folds for large datasets, 5 for small. Reduces fit count."""
    n_folds = 3 if n_train > LARGE_THRESHOLD else 5
    return StratifiedKFold(
        n_splits=n_folds, shuffle=True, random_state=SEED)

def train_classical(name, X_tr, X_val, y_tr, y_val):
    configs = get_clf_configs(len(X_tr))
    cfg     = configs[name]
    cv      = get_cv(len(X_tr))

    if cfg['search'] == 'random':
        searcher = RandomizedSearchCV(
            cfg['estimator'], cfg['param_grid'],
            n_iter=cfg.get('n_iter', 8),
            scoring='f1_macro', cv=cv,
            n_jobs=-1, refit=True,
            random_state=SEED)
    else:
        searcher = GridSearchCV(
            cfg['estimator'], cfg['param_grid'],
            scoring='f1_macro', cv=cv,
            n_jobs=-1, refit=True)

    t0 = time.time()
    searcher.fit(X_tr, y_tr)
    elapsed = round(time.time() - t0, 1)

    val_f1 = f1_score(y_val, searcher.predict(X_val), average='macro')
    print(f"    {name}: params={searcher.best_params_} "
          f"val_F1={val_f1:.4f}  ({elapsed}s)")
    return searcher.best_estimator_, searcher.best_params_, elapsed

In [ ]:
def record_result(dataset, clf_name, condition, fspace,
                  model, X_te, y_te, best_params=None, train_sec=None):
    y_pred = model.predict(X_te) if not isinstance(model, np.ndarray) \
             else model  # allow passing y_pred directly for FTT
    entry = {
        'dataset':      dataset,
        'classifier':   clf_name,
        'condition':    condition,   # 'original' | 'smote'
        'feature_space': fspace,     # 'full' | 'pca'
        'f1_macro':     round(f1_score(y_te, y_pred, average='macro'), 4),
        'f1_class0':    round(f1_score(y_te, y_pred, pos_label=0), 4),
        'f1_class1':    round(f1_score(y_te, y_pred, pos_label=1), 4),
        'accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'precision_macro': round(precision_score(y_te, y_pred,
                                  average='macro', zero_division=0), 4),
        'recall_macro': round(recall_score(y_te, y_pred,
                               average='macro'), 4),
        'best_params':  str(best_params),
        'train_sec':    train_sec,
        'y_pred':       y_pred.tolist()
    }
    RESULTS.append(entry)
    print(f"    → F1={entry['f1_macro']} Acc={entry['accuracy']}")
    return entry

In [ ]:
import json, os

RESULTS      = []
YTEST_MAP    = {}
DATASETS_STORE = {}
CHECKPOINT_FILE = 'checkpoint.json'

# -------------------------------------------------------
# Load any existing checkpoint so you can resume
# -------------------------------------------------------
def load_checkpoint():
    global RESULTS, YTEST_MAP
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
        RESULTS = data.get('results', [])
        # Rebuild YTEST_MAP from saved arrays
        for k, v in data.get('ytest_map', {}).items():
            YTEST_MAP[k] = np.array(v)
        print(f"Resumed: {len(RESULTS)} runs already complete.")
    else:
        print("No checkpoint found — starting fresh.")

def save_checkpoint():
    data = {
        'results': RESULTS,
        'ytest_map': {k: v.tolist() for k, v in YTEST_MAP.items()}
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f)

def already_done(ds, clf, condition, fspace):
    """Return True if this exact run is already in RESULTS."""
    for r in RESULTS:
        if (r['dataset'] == ds and r['classifier'] == clf
                and r['condition'] == condition
                and r['feature_space'] == fspace):
            return True
    return False

# Call this at the top of your notebook before running anything
load_checkpoint()

# -------------------------------------------------------
# Subsample limits per dataset for speed
# -------------------------------------------------------
TRAIN_LIMITS = {
    'credit_card': 40_000,   # LR+RF use this; SVM uses 20k
    'nslkdd':      20_000,     # use full dataset (LinearSVC handles it)
}

SVM_LIMITS = {
    'credit_card': 20_000,
    'nslkdd':      20_000,
}

def get_train_sample(ds_name, clf_name, X_tr, y_tr):
    """Return (X_use, y_use) — subsampled if needed."""
    if clf_name == 'SVM' and ds_name in SVM_LIMITS:
        limit = SVM_LIMITS[ds_name]
    else:
        limit = TRAIN_LIMITS.get(ds_name)
    if limit and len(X_tr) > limit:
        idx = np.random.RandomState(SEED).choice(
            len(X_tr), limit, replace=False)
        print(f"      ({clf_name} subsample: {limit:,} rows)")
        return X_tr[idx], y_tr[idx]
    return X_tr, y_tr


def run_dataset(ds_name, X, y, X_te_fixed=None, y_te_fixed=None):
    print(f"\n{'='*55}")
    print(f"DATASET: {ds_name.upper()}")
    print(f"{'='*55}")
    DATASETS_STORE[ds_name] = (X.copy(), y.copy())

    for condition in ['original', 'smote']:
        for fspace in ['full', 'pca']:
            print(f"\n  [{condition.upper()} | {fspace.upper()}]")

            # Check if ALL 3 classifiers are already done
            done = [already_done(ds_name, c, condition, fspace)
                    for c in ['LR','RF','SVM']]
            if all(done):
                print("    All 3 classifiers already done — skipping splits.")
                continue

            X_tr, X_val, X_te, y_tr, y_val, y_te, scaler, pca = \
                make_splits(X, y,
                    apply_smote=(condition=='smote'),
                    apply_pca=(fspace=='pca'),
                    X_te_fixed=X_te_fixed,
                    y_te_fixed=y_te_fixed)

            key = f"{ds_name}_{condition}_{fspace}"
            YTEST_MAP[key] = y_te.copy()

            for clf_name in ['LR', 'RF', 'SVM']:
                # Skip if already checkpointed
                if already_done(ds_name, clf_name, condition, fspace):
                    print(f"    {clf_name}: already done — skipped.")
                    continue

                print(f"    Training {clf_name}...")
                X_use, y_use = get_train_sample(
                    ds_name, clf_name, X_tr, y_tr)
                model, params, secs = train_classical(
                    clf_name, X_use, X_val, y_use, y_val)
                record_result(ds_name, clf_name, condition,
                              fspace, model, X_te, y_te, params, secs)

                # Save after every single run
                save_checkpoint()
                print(f"    Checkpoint saved ({len(RESULTS)} runs total).")


# -------------------------------------------------------
# Run all datasets
# -------------------------------------------------------
X_ph, y_ph = load_phishing(path=os.path.join(DATA_DIR, 'Training Dataset.arff'))
run_dataset('phishing', X_ph, y_ph)

X_ktr, y_ktr, X_kte, y_kte = load_nslkdd(
    train_path=os.path.join(DATA_DIR, 'KDDTrain+.txt'),
    test_path=os.path.join(DATA_DIR, 'KDDTest+.txt'))
run_dataset('nslkdd', X_ktr, y_ktr, X_kte, y_kte)

X_sp, y_sp = load_spambase(path=os.path.join(DATA_DIR, 'spambase.data'))
run_dataset('spambase', X_sp, y_sp)

X_cc, y_cc = load_creditcard(path=os.path.join(DATA_DIR, 'creditcard.csv'))
run_dataset('credit_card', X_cc, y_cc)

# -------------------------------------------------------
# Final save
# -------------------------------------------------------
df_res = pd.DataFrame([{k:v for k,v in r.items() if k!='y_pred'}
                        for r in RESULTS])
df_res.to_csv('results.csv', index=False)
with open('results_with_preds.json','w') as f:
    json.dump(RESULTS, f)
print(f"\nDone. {len(RESULTS)} runs saved.")

In [ ]:
# Install (robust version)
!pip install --upgrade pip -q
!pip install git+https://github.com/yandex-research/rtdl -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Safe import with clear error if something still breaks
try:
    import rtdl
except ImportError as e:
    raise ImportError(
        "rtdl failed to install. Restart the runtime and run the install cell again."
    ) from e

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def build_ftt(n_features):
    """Smaller model — faster, still effective on tabular data."""
    return rtdl.FTTransformer.make_baseline(
        n_num_features=n_features,
        cat_cardinalities=None,
        d_token=32,          # was 64 — 32 is sufficient for these datasets
        n_blocks=2,          # was 3 — 2 blocks converge faster
        attention_dropout=0.1,
        ffn_dropout=0.1,
        ffn_d_hidden=32 * 4, # Added ffn_d_hidden
        residual_dropout=0.0,
        d_out=1
    ).to(DEVICE)

In [ ]:
def train_ftt(X_tr, y_tr, X_val, y_val,
               ds_name='', lr=1e-4, weight_decay=1e-5):

    n_feat    = X_tr.shape[1]
    model     = build_ftt(n_feat)
    opt       = torch.optim.AdamW(model.parameters(),
                                   lr=lr, weight_decay=weight_decay)
    ep_max, patience = 40, 5      # hard cap — was 100/10
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep_max)
    amp_scaler = GradScaler()

    pos_w = torch.tensor(
        [(y_tr==0).sum() / max((y_tr==1).sum(), 1)],
        dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    Xtr_t  = torch.tensor(X_tr,  dtype=torch.float32).to(DEVICE)
    ytr_t  = torch.tensor(y_tr.astype(float), dtype=torch.float32).to(DEVICE)
    Xval_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr_t, ytr_t),
        batch_size=1024,          # was 256 — larger batch = fewer steps
        shuffle=True, drop_last=False)

    best_f1, best_state, stall = 0.0, None, 0
    t0 = time.time()

    for ep in range(ep_max):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            with autocast():
                loss = criterion(model(Xb, None).squeeze(-1), yb)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()
        sched.step()

        model.eval()
        with torch.no_grad():
            vpred = (torch.sigmoid(
                model(Xval_t, None).squeeze(-1)) > 0.5).cpu().numpy()
        val_f1 = f1_score(y_val, vpred, average='macro')

        if val_f1 > best_f1:
            best_f1    = val_f1
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            stall = 0
        else:
            stall += 1
        if stall >= patience:
            break

    model.load_state_dict(best_state)
    elapsed = round(time.time() - t0, 1)
    print(f"    ep={ep+1} | best_val_F1={best_f1:.4f} | {elapsed}s")
    return model, best_f1, elapsed


def predict_ftt(model, X_te):
    model.eval()
    with torch.no_grad():
        logits = model(
            torch.tensor(X_te, dtype=torch.float32).to(DEVICE), None
        ).squeeze(-1)
    return (torch.sigmoid(logits) > 0.5).cpu().numpy().astype(int)

In [ ]:
# Cache: {(ds_name, condition, fspace): (X_tr, X_val, X_te, y_tr, y_val, y_te)}
SPLIT_CACHE = {}

def cache_splits(ds_name, X_raw, y_raw, X_te_fixed=None, y_te_fixed=None):
    """
    Run make_splits for all 4 conditions and store results.
    SMOTE and PCA run only once each — not repeated per classifier.
    """
    print(f"  Caching splits for {ds_name}...")
    kw = {}
    if X_te_fixed is not None:
        kw = {'X_te_fixed': X_te_fixed, 'y_te_fixed': y_te_fixed}
    for condition in ['original', 'smote']:
        for fspace in ['full', 'pca']:
            key = (ds_name, condition, fspace)
            if key in SPLIT_CACHE:
                continue  # already cached
            print(f"    Splitting: {condition} | {fspace}...", end=' ')
            t0 = time.time()
            X_tr, X_val, X_te, y_tr, y_val, y_te, sc, pca = make_splits(
                X_raw, y_raw,
                apply_smote=(condition == 'smote'),
                apply_pca=(fspace == 'pca'),
                **kw)
            SPLIT_CACHE[key] = (X_tr, X_val, X_te, y_tr, y_val, y_te)
            # Also update YTEST_MAP in case it's needed
            YTEST_MAP[f"{ds_name}_{condition}_{fspace}"] = y_te.copy()
            print(f"done in {time.time()-t0:.1f}s "
                  f"(train={len(X_tr):,} rows)")

# Pre-cache all datasets — run this cell BEFORE the FTT loop
cache_splits('phishing',    X_ph,  y_ph)
cache_splits('nslkdd',      X_ktr, y_ktr, X_kte, y_kte)
cache_splits('spambase',    X_sp,  y_sp)
cache_splits('credit_card', X_cc,  y_cc)
print("All splits cached.")

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# Epoch / patience schedule by training set size
def get_ftt_schedule(n_train):
    if n_train < 8_000:
        return 60, 7    # small: 60 epochs, patience 7
    elif n_train < 40_000:
        return 80, 8    # medium
    else:
        return 100, 10  # large

def train_ftt(X_tr, y_tr, X_val, y_val,
               ds_name='', n_epochs=None, lr=1e-4,
               batch_sz=256, weight_decay=1e-5, patience=None):

    n_feat = X_tr.shape[1]
    model  = build_ftt(n_feat)
    opt    = torch.optim.AdamW(model.parameters(),
                                lr=lr, weight_decay=weight_decay)

    # Adaptive schedule
    ep_max, pat = get_ftt_schedule(len(X_tr))
    if n_epochs is not None: ep_max = n_epochs
    if patience  is not None: pat   = patience

    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep_max)
    scaler_amp = GradScaler()   # for mixed precision

    pos_w = torch.tensor(
        [(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
        dtype=torch.float32).to(DEVICE)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)

    def to_t(*arrs):
        return [torch.tensor(a, dtype=torch.float32).to(DEVICE)
                for a in arrs]

    Xtr_t, ytr_t   = to_t(X_tr, y_tr.astype(float))
    Xval_t, yval_t = to_t(X_val, y_val.astype(float))
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_sz, shuffle=True, drop_last=False,
        pin_memory=False)

    best_f1, best_state, stall = 0.0, None, 0
    t0 = time.time()

    for ep in range(ep_max):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)   # faster than zero_grad()
            # Mixed precision forward pass
            with autocast():
                logits = model(Xb, None).squeeze(-1)
                loss   = criterion(logits, yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
        sched.step()

        # Validation (no autocast needed — fast anyway)
        model.eval()
        with torch.no_grad():
            vlogits = model(Xval_t, None).squeeze(-1)
            vpred   = (torch.sigmoid(vlogits) > 0.5).cpu().numpy()
        val_f1 = f1_score(y_val, vpred, average='macro')

        if val_f1 > best_f1:
            best_f1    = val_f1
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            stall = 0
        else:
            stall += 1
        if stall >= pat:
            break   # early stop

    elapsed = round(time.time() - t0, 1)
    model.load_state_dict(best_state)
    print(f"  FTT: ep={ep+1}/{ep_max} best_val_F1={best_f1:.4f} ({elapsed}s)")
    return model, best_f1, elapsed


def predict_ftt(model, X_te):
    model.eval()
    Xte_t = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        logits = model(Xte_t, None).squeeze(-1)
    return (torch.sigmoid(logits) > 0.5).cpu().numpy().astype(int)

In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler
import time # Import time if not already at top of scope

# ── PART 1: Fast training wrapper — drop this in above your loop ───────────────
def train_ftt_fast(X_tr, y_tr, X_val, y_val, ds_name='',
                   max_epochs=50,       # was probably 100-200 — halve it
                   patience=5,          # stop early if val loss stalls
                   batch_size=512,      # larger batches → fewer steps
                   lr=1e-3):
    """
    Wraps your existing FTT components with:
      • AMP mixed precision  (fp16 forward, fp32 grad)  ~2× faster on T4/A100
      • Aggressive early stopping                        cuts wasted epochs
      • pin_memory + persistent DataLoader workers       faster CPU→GPU transfer
      • torch.compile (PyTorch ≥ 2.0)                   ~10-20% extra
    Replace the body with your actual model/optimiser wiring.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ── Convert to tensors once, keep on CPU, let DataLoader pin ──────────────
    X_t = torch.tensor(X_tr,  dtype=torch.float32)
    y_t = torch.tensor(y_tr.astype(float),  dtype=torch.float32) # Changed to float32
    Xv  = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv  = torch.tensor(y_val.astype(float), dtype=torch.float32).to(device) # Changed to float32

    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader  = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,          # async CPU→GPU copy
        num_workers=2,            # parallel prefetch (Colab has 2 vCPUs)
        persistent_workers=True,  # reuse workers across epochs
        drop_last=False,
    )

    # ── Build your model exactly as before ────────────────────────────────────
    model = build_ftt(X_tr.shape[1]).to(device) # Changed from build_ftt_model to build_ftt

    # Optional: compile if PyTorch ≥ 2.0 (free ~15% speedup, silent fallback)
    if hasattr(torch, 'compile'):
        model = torch.compile(model, mode='reduce-overhead')

    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    # Cosine LR decay — converges in fewer epochs than flat LR
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=max_epochs)

    # Calculate pos_weight for BCEWithLogitsLoss
    pos_w = torch.tensor(
        [(y_tr==0).sum() / max((y_tr==1).sum(), 1)],
        dtype=torch.float32).to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w) # Changed from CrossEntropyLoss

    scaler    = GradScaler()          # AMP loss scaler

    best_val, best_state, no_improve = float('inf'), None, 0
    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device, non_blocking=True), \
                     yb.to(device, non_blocking=True)
            optimiser.zero_grad(set_to_none=True)   # faster than zero_grad()

            with autocast():                         # fp16 forward pass
                loss = criterion(model(Xb, None).squeeze(-1), yb) # Adjusted for BCEWithLogitsLoss and build_ftt output

            scaler.scale(loss).backward()
            scaler.step(optimiser)
            scaler.update()

        scheduler.step()

        # ── Early stopping on validation loss ─────────────────────────────────
        model.eval()
        with torch.no_grad(), autocast():
            val_loss = criterion(model(Xv, None).squeeze(-1), yv).item() # Adjusted for BCEWithLogitsLoss and build_ftt output

        if val_loss < best_val - 1e-4:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in
                          model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"    early stop @ epoch {epoch+1} "
                      f"(val_loss={best_val:.4f})")
                break

    model.load_state_dict(best_state)
    return model, None, time.time() - t0

# ── PART 2: Optimised outer loop ──────────────────────────────────────────────
FTT_MAX_ROWS = {
    'credit_card': 10_000,
    'nslkdd':      10_000,
    'phishing':    None,
    'spambase':    None,
}

# Pre-fetch all splits once so the loop never re-splits
def ensure_splits(ds_name):
    for condition in ['original', 'smote']:
        key = (ds_name, condition, 'full')
        if key not in SPLIT_CACHE:
            X_raw, y_raw = DATASETS_STORE[ds_name]
            kw = ({'X_te_fixed': X_kte, 'y_te_fixed': y_kte}
                  if ds_name == 'nslkdd' else {})
            cache_splits(ds_name, X_raw, y_raw, **kw)

for ds_name in ['phishing', 'nslkdd', 'spambase', 'credit_card']:
    print(f"\n{'='*45}\nFTT — {ds_name.upper()}\n{'='*45}")
    ensure_splits(ds_name)  # batch the I/O once per dataset

    for condition in ['original', 'smote']:
        if already_done(ds_name, 'FTT', condition, 'full'):
            print(f"  [{condition}|full] already done — skip.")
            continue

        X_tr, X_val, X_te, y_tr, y_val, y_te = SPLIT_CACHE[
            (ds_name, condition, 'full')]

        max_r = FTT_MAX_ROWS.get(ds_name)
        if max_r and len(X_tr) > max_r:
            rng   = np.random.RandomState(SEED)
            idx   = rng.choice(len(X_tr), max_r, replace=False)
            X_ftt, y_ftt = X_tr[idx], y_tr[idx]
            print(f"  [{condition}|full] subsample → {max_r:,} rows")
        else:
            X_ftt, y_ftt = X_tr, y_tr

        print(f"  [{condition}|full] FTT training "
              f"({len(X_ftt):,} rows × {X_ftt.shape[1]} features)...")

        ftt_model, _, secs = train_ftt_fast(     # ← new wrapper below
            X_ftt, y_ftt, X_val, y_val, ds_name=ds_name)

        y_pred = predict_ftt(ftt_model, X_te)
        record_result(ds_name, 'FTT', condition, 'full',
                      y_pred, X_te, y_te,
                      best_params={'d_token': 32, 'n_blocks': 2},
                      train_sec=secs)

        # Checkpoint only on change, not every iteration
        save_checkpoint()
        print(f"  Saved. Total runs: {len(RESULTS)}")

# ── Final save (unchanged) ─────────────────────────────────────────────────────
df_all = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred'}
                        for r in RESULTS])
df_all.to_csv('results_full.csv', index=False)
with open('results_with_preds.json', 'w') as f:
    json.dump(RESULTS, f)
print(f"\nAll done. {len(RESULTS)} runs saved.")

In [ ]:
import json
with open('results_with_preds.json') as f:
    RESULTS = json.load(f)

def get_preds(ds, clf, condition, fspace='full'):
    for r in RESULTS:
        if (r['dataset']==ds and r['classifier']==clf
                and r['condition']==condition
                and r['feature_space']==fspace):
            return np.array(r['y_pred'])
    return None

def get_yte(ds, condition, fspace='full'):
    return YTEST_MAP.get(f"{ds}_{condition}_{fspace}")

In [ ]:
from itertools import combinations

CLFS   = ['LR','RF','SVM','FTT']
DS     = ['phishing','nslkdd','spambase','credit_card']
CONDS  = ['original','smote']
PAIRS  = list(combinations(CLFS, 2))  # 6 pairs

mc_rows = []
for ds in DS:
    for cond in CONDS:
        y_te = get_yte(ds, cond)
        if y_te is None:
            continue
        for (a, b) in PAIRS:
            ya = get_preds(ds, a, cond)
            yb = get_preds(ds, b, cond)
            if ya is None or yb is None:
                continue
            # 2×2 contingency table
            #         B correct   B wrong
            # A correct   n00        b
            # A wrong      c        n11
            n00 = np.sum((ya==y_te) & (yb==y_te))
            b_  = np.sum((ya==y_te) & (yb!=y_te))
            c_  = np.sum((ya!=y_te) & (yb==y_te))
            n11 = np.sum((ya!=y_te) & (yb!=y_te))
            table = np.array([[n00, b_], [c_, n11]])
            # McNemar with continuity correction (Yates)
            result = mc_test(table, exact=False, correction=True)
            chi2   = round(result.statistic, 3)
            p_raw  = round(result.pvalue, 5)
            mc_rows.append({
                'dataset': ds, 'condition': cond,
                'clf_a': a,    'clf_b': b,
                'b':    int(b_), 'c': int(c_),
                'chi2': chi2,   'p_raw': p_raw,
                'f1_a': round(f1_score(y_te, ya, average='macro'),4),
                'f1_b': round(f1_score(y_te, yb, average='macro'),4)
            })

mc_df = pd.DataFrame(mc_rows)
N_TESTS = len(mc_df)
print(f"Total McNemar tests: {N_TESTS}")

# Bonferroni correction
mc_df['p_bonf']    = (mc_df['p_raw'] * N_TESTS).clip(upper=1.0)
mc_df['p_bonf']    = mc_df['p_bonf'].round(5)
mc_df['sig_raw']   = mc_df['p_raw'] < 0.05
mc_df['sig_bonf']  = mc_df['p_bonf'] < 0.05

# Summary
n_sig_raw  = mc_df['sig_raw'].sum()
n_sig_bonf = mc_df['sig_bonf'].sum()
print(f"Significant (raw p<0.05):       {n_sig_raw}/{N_TESTS}")
print(f"Significant (Bonferroni):        {n_sig_bonf}/{N_TESTS}")
print(f"NOT significant (Bonferroni):    {N_TESTS-n_sig_bonf}/{N_TESTS}")
mc_df.to_csv('mcnemar_results.csv', index=False)


In [ ]:
def cohens_d(scores_a, scores_b):
    a, b = np.array(scores_a), np.array(scores_b)
    pooled = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return round(abs(a.mean()-b.mean()) / pooled, 3) if pooled>0 else 0.0

def interp_d(d):
    if d < 0.2: return 'negligible'
    if d < 0.5: return 'small'
    if d < 0.8: return 'medium'
    return 'large'

cohens_rows = []
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

clf_objects = {
    'LR':  LogisticRegression(C=1, max_iter=2000,
                               solver='lbfgs', random_state=SEED),
    'RF':  RandomForestClassifier(n_estimators=200,
                                   random_state=SEED, n_jobs=-1),
    'SVM': SVC(C=1, kernel='rbf', gamma='scale', random_state=SEED)
}

for ds in DS:
    X_raw, y_raw = DATASETS_STORE[ds]
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    # Subsample for SVM CV on large datasets
    if ds in ('credit_card','nslkdd') and len(X_sc) > 50000:
        idx  = np.random.RandomState(SEED).choice(
            len(X_sc), 50000, replace=False)
        Xs, ys = X_sc[idx], y_raw[idx]
    else:
        Xs, ys = X_sc, y_raw

    clf_scores = {}
    for name, clf in clf_objects.items():
        scores = cross_val_score(clf, Xs, ys,
            cv=cv5, scoring='f1_macro', n_jobs=-1)
        clf_scores[name] = scores
        print(f"  {ds} {name}: {scores.round(4)} mean={scores.mean():.4f}")

    for (a,b) in combinations(['LR','RF','SVM'], 2):
        d = cohens_d(clf_scores[a], clf_scores[b])
        cohens_rows.append({
            'dataset': ds, 'clf_a': a, 'clf_b': b,
            'cohens_d': d, 'interpretation': interp_d(d)
        })

pd.DataFrame(cohens_rows).to_csv('cohens_d.csv', index=False)

In [ ]:
df_r = pd.read_csv('results_full.csv')
pivot = df_r[df_r.feature_space=='full'].pivot_table(
    index=['dataset','condition'],
    columns='classifier',
    values='f1_macro'
).reset_index()
pivot.columns = [str(c) for c in pivot.columns]
print(pivot.to_string(index=False))
pivot.to_csv('table2_results.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np

def plot_heatmap(mc_df, condition='original', corrected=True,
                 save_path=None):
    p_col  = 'p_bonf' if corrected else 'p_raw'
    pairs  = [f"{a} vs {b}" for (a,b) in PAIRS]
    ds_labels = {'phishing':'Phishing\n(UCI)',
                 'nslkdd':'NSL-KDD',
                 'spambase':'Spambase',
                 'credit_card':'Credit Card\nFraud'}
    sub  = mc_df[mc_df.condition==condition]
    mat  = np.ones((len(DS), len(PAIRS))) * np.nan
    for i, ds in enumerate(DS):
        for j, (a,b) in enumerate(PAIRS):
            row = sub[(sub.dataset==ds)&(sub.clf_a==a)&(sub.clf_b==b)]
            if len(row):
                mat[i,j] = row[p_col].values[0]

    fig, ax = plt.subplots(figsize=(10, 3.6))
    cmap = sns.diverging_palette(15, 130, s=80, l=55, as_cmap=True)
    im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=0.2,
                   aspect='auto')

    # Annotate cells
    alpha = 0.00104 if corrected else 0.05
    for i in range(len(DS)):
        for j in range(len(PAIRS)):
            v = mat[i,j]
            if np.isnan(v): continue
            sig = v < alpha
            txt = f"{v:.4f}"
            fc  = 'white' if v < 0.05 else '#2C2C2A'
            fw  = 'bold' if sig else 'normal'
            ax.text(j, i, txt, ha='center', va='center',
                    fontsize=8.5, color=fc, fontweight=fw)
            if sig:
                ax.add_patch(plt.Rectangle(
                    (j-0.5,i-0.5), 1, 1,
                    fill=False, edgecolor='#27500A', lw=2))

    ax.set_xticks(range(len(PAIRS)))
    ax.set_xticklabels(pairs, fontsize=9, rotation=30, ha='right')
    ax.set_yticks(range(len(DS)))
    ax.set_yticklabels([ds_labels[d] for d in DS], fontsize=9)
    title_sfx = 'Bonferroni-corrected' if corrected else 'uncorrected'
    ax.set_title(
        f"McNemar p-values ({title_sfx}) — "
        f"{condition.capitalize()} training condition",
        fontsize=10, pad=10)
    cb = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cb.set_label('p-value', fontsize=8)
    cb.ax.axhline(y=alpha, color='#27500A', lw=1, ls='--')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_heatmap(mc_df, 'original', corrected=True,
             save_path='fig1a_heatmap_original.pdf')
plot_heatmap(mc_df, 'smote',    corrected=True,
             save_path='fig1b_heatmap_smote.pdf')


In [ ]:
def plot_bump_chart(df_res, condition='original',
                    fspace='full', save_path=None):
    sub = df_res[(df_res.condition==condition) &
                 (df_res.feature_space==fspace)].copy()
    # Compute ranks per dataset (1=best)
    sub['rank'] = sub.groupby('dataset')['f1_macro'].rank(
        ascending=False, method='min').astype(int)

    ds_order = ['phishing','nslkdd','spambase','credit_card']
    ds_labels = ['Phishing','NSL-KDD','Spambase','Credit Card']
    clf_colors = {'LR':'#534AB7','RF':'#0F6E56',
                  'SVM':'#993C1D','FTT':'#185FA5'}
    clf_markers = {'LR':'o','RF':'s','SVM':'^','FTT':'D'}

    fig, ax = plt.subplots(figsize=(7, 4))
    x_pos = {ds:i for i,ds in enumerate(ds_order)}

    for clf in ['LR','RF','SVM','FTT']:
        xs, ys, f1s = [], [], []
        for ds in ds_order:
            row = sub[(sub.dataset==ds)&(sub.classifier==clf)]
            if len(row):
                xs.append(x_pos[ds])
                ys.append(row['rank'].values[0])
                f1s.append(row['f1_macro'].values[0])
        if xs:
            ax.plot(xs, ys, '-', color=clf_colors[clf], lw=2, alpha=0.85)
            for x,y,f in zip(xs,ys,f1s):
                ax.scatter(x, y, color=clf_colors[clf],
                           s=70, zorder=5,
                           marker=clf_markers[clf],
                           label=clf if x==0 else '')
                ax.text(x, y-0.15, f'{f:.3f}',
                        ha='center', va='bottom',
                        fontsize=7.5, color=clf_colors[clf])

    ax.set_xticks(range(len(ds_order)))
    ax.set_xticklabels(ds_labels, fontsize=9)
    ax.set_yticks([1,2,3,4])
    ax.set_yticklabels(['1st','2nd','3rd','4th'], fontsize=9)
    ax.invert_yaxis()
    ax.set_ylabel('Rank by macro F1', fontsize=9)
    ax.set_title(
        f'Classifier ranking stability across domains '
        f'({condition}, {fspace} features)', fontsize=10)
    handles = [plt.Line2D([0],[0], color=clf_colors[c],
               marker=clf_markers[c], lw=2, ms=7, label=c)
               for c in ['LR','RF','SVM','FTT']]
    ax.legend(handles=handles, fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3, lw=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_bump_chart(df_all, 'original', 'full',
                save_path='fig2_rank_chart.pdf')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

# --- Visualization 1: Cohen's d Effect Size Heatmap ---

def plot_cohens_d_heatmap(df, save_path=None):
    df['pair'] = df.apply(lambda row: f"{row['clf_a']} vs {row['clf_b']}", axis=1)
    pivot_df = df.pivot_table(index='dataset', columns='pair', values='cohens_d')

    ds_labels = {'phishing':'Phishing (UCI)',
                 'nslkdd':'NSL-KDD',
                 'spambase':'Spambase',
                 'credit_card':'Credit Card Fraud'}
    pivot_df = pivot_df.rename(index=ds_labels)

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot_df, annot=True, cmap='viridis', fmt=".2f", linewidths=.5, ax=ax)
    ax.set_title("Cohen's d Effect Size between Classifier Pairs (Cross-validation)")
    ax.set_xlabel("Classifier Pair")
    ax.set_ylabel("Dataset")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


# --- Visualization 2: Macro F1-score Comparison Bar Chart ---

def plot_f1_macro_comparison(df, save_path=None):
    df_plot = df.copy()
    df_plot['Condition_FeatureSpace'] = df_plot['condition'] + ' / ' + df_plot['feature_space']

    clf_order = ['LR', 'RF', 'SVM', 'FTT']
    df_plot['classifier'] = pd.Categorical(df_plot['classifier'], categories=clf_order, ordered=True)

    ds_labels = {'phishing':'Phishing (UCI)',
                 'nslkdd':'NSL-KDD',
                 'spambase':'Spambase',
                 'credit_card':'Credit Card Fraud'}
    df_plot['dataset'] = df_plot['dataset'].replace(ds_labels)

    g = sns.catplot(
        data=df_plot,
        x='Condition_FeatureSpace',
        y='f1_macro',
        hue='classifier',
        col='dataset',
        col_wrap=2,
        kind='bar',
        height=4, aspect=1.2,
        palette='deep',
        errorbar=None,
        col_order=[ds_labels['phishing'], ds_labels['nslkdd'], ds_labels['spambase'], ds_labels['credit_card']]
    )
    g.set_axis_labels("Condition / Feature Space", "Macro F1-score")
    g.set_titles("Dataset: {col_name}")
    g.set_xticklabels(rotation=30, ha='right')
    g.fig.suptitle("Macro F1-score Comparison Across Datasets, Conditions, and Feature Spaces", y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


# --- Visualization 3: Confusion Matrix for Credit Card Fraud ---

def plot_confusion_matrix(y_true, y_pred, title, class_names=['Class 0', 'Class 1'], save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Generate Cohen's d Effect Size Heatmap
cohens_df = pd.read_csv('cohens_d.csv')
plot_cohens_d_heatmap(cohens_df, save_path='fig3_cohens_d_heatmap.pdf')

In [ ]:
# Generate Macro F1-score Comparison Bar Chart
df_all = pd.read_csv('results_full.csv')
plot_f1_macro_comparison(df_all, save_path='fig4_f1_macro_bars.pdf')

In [ ]:
# Generate Confusion Matrix for Credit Card Fraud (FTT, SMOTE, Full Features)
# Ensure get_yte and get_preds functions are available (from cell wjWza4mlxVUr)

dataset_name = 'credit_card'
classifier_name = 'FTT'
condition = 'smote'
feature_space = 'full'

y_true_cc = get_yte(dataset_name, condition, feature_space)
y_pred_cc = get_preds(dataset_name, classifier_name, condition, feature_space)

if y_true_cc is not None and y_pred_cc is not None:
    plot_confusion_matrix(y_true_cc, y_pred_cc,
                          f"Confusion Matrix: {dataset_name.replace('_', ' ').title()} - {classifier_name} ({condition}, {feature_space})",
                          class_names=['Normal', 'Fraud'],
                          save_path='fig5_creditcard_ftt_smote_cm.pdf')
else:
    print(f"Data not found for {dataset_name}, {classifier_name}, {condition}, {feature_space}")

### McNemar's Test Results Summary (Bonferroni Corrected)

This table summarizes the results of McNemar's test, indicating whether the performance difference between two classifiers is statistically significant after applying Bonferroni correction. `True` indicates a significant difference (p < 0.05).

In [ ]:
mc_summary = mc_df[['dataset', 'condition', 'clf_a', 'clf_b', 'p_bonf', 'sig_bonf']]
# Pivot the table to show significance for each pair per dataset/condition
mc_pivot = mc_summary.pivot_table(
    index=['dataset', 'condition'],
    columns=['clf_a', 'clf_b'],
    values='sig_bonf'
)

# Flatten the column names for better readability
mc_pivot.columns = ['_'.join(col).strip() for col in mc_pivot.columns.values]

display(mc_pivot)
mc_pivot.to_csv('mcnemar_significant_summary.csv')
print("McNemar's significant summary saved to 'mcnemar_significant_summary.csv'")